In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
from sklearn.preprocessing import StandardScaler, MaxAbsScaler

In [2]:
df = pd.read_parquet("/kaggle/input/sazerac-dataset-preprocessed/sales_features.parquet")

In [3]:
df.sort_values(["zipcode", "date"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [4]:
to_drop = [
    *[c for c in df.columns if c.startswith("sale_gallons")],
    "category_count", "unique_category_names", "category_name_count",
    "itemno_count"
]
df.drop(columns=to_drop, inplace=True)

In [5]:
def binary_encode(df, col):
    vals = df[col].fillna(0).astype(int).values
    n_bits = int(math.ceil(np.log2(vals.max() + 1)))
    bit_cols = []
    for i in range(n_bits):
        bit_col = f"{col}_bit{i}"
        df[bit_col] = ((vals >> i) & 1).astype(int)
        bit_cols.append(bit_col)
    return bit_cols

cat_bits = binary_encode(df, "unique_categories")
item_bits = binary_encode(df, "unique_items")
df.drop(columns=["unique_categories", "unique_items"], inplace=True)


In [6]:
IDENTIFIERS = ["date", "zipcode"]
STATIC_NUM = [
    "lon", "lat",
    *cat_bits, *item_bits,
    "store_avg_sales", "store_size", "city_avg_sales", "county_avg_sales",
    "store_avg_transactions", "store_to_city_sales_ratio",
    "store_to_county_sales_ratio", "store_avg_items",
    "items_per_transaction_ratio",
    # calendar flags & encodings
    "day_of_week", "month", "quarter", "year",
    "day_of_month", "week_of_year",
    "is_weekend", "is_holiday", "is_day_before_holiday", "is_day_after_holiday",
    "days_to_nearest_holiday",
    "month_sin", "month_cos", "day_of_month_sin", "day_of_month_cos"
]
DYNAMIC = [
    # daily sales aggregates
    *[c for c in df.columns if c.startswith("sale_bottles")],
    *[c for c in df.columns if c.startswith("sale_dollars")],
    *[c for c in df.columns if c.startswith("sale_liters")],
    "transaction_count", "unique_transactions",
    # pack & bottle sizes
    *[c for c in df.columns if c.startswith("pack_")],
    *[c for c in df.columns if c.startswith("bottle_volume_ml")],
    "avg_items_per_transaction",
    # lags & rolling
    *[c for c in df.columns if "lag_" in c],
    *[c for c in df.columns if "rolling_" in c],
    *[c for c in df.columns if "momentum_" in c],
    *[c for c in df.columns if "avg_days_between_purchases" in c],
    "days_since_last_purchase", "sale_dollars_decrease_prev",
    "sale_dollars_decrease_7d_avg", "sale_dollars_decrease_30d_avg",
    "sale_dollars_significant_decrease", "consecutive_decrease"
]
TARGET = "sale_dollars"

In [7]:
STATIC_NUM = list(dict.fromkeys(STATIC_NUM))
DYNAMIC    = list(dict.fromkeys(DYNAMIC))

In [8]:
dupes = df.columns[df.columns.duplicated()]
print("Duplicate column names:", dupes.unique())

Duplicate column names: Index([], dtype='object')


In [9]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [10]:
df[STATIC_NUM] = df[STATIC_NUM].astype(float)

df.loc[:, STATIC_NUM] = df[STATIC_NUM].fillna(0)
for c in DYNAMIC:
    df[c] = df[c].ffill().fillna(0)

for c in DYNAMIC:
    lo, hi = np.nanpercentile(df[c], [1, 99])
    df[c] = df[c].clip(lo, hi)

df.loc[:, STATIC_NUM] = StandardScaler().fit_transform(df[STATIC_NUM])
for c in DYNAMIC:
    df[c] = MaxAbsScaler().fit_transform(df[[c]])

In [11]:
class SlidingWindowDataset(Dataset):
    def __init__(self, df, dyn_cols, stat_cols, target, seq_len=30, horizon=30):
        self.seq_len, self.horizon = seq_len, horizon
        self.dyn_cols, self.stat_cols = dyn_cols, stat_cols
        seqs, stats, tgts = [], [], []
        for _, grp in df.groupby("zipcode"):
            Xd = grp[dyn_cols].values
            Xs = grp[stat_cols].values
            y  = grp[target].values
            n = len(grp)
            for i in range(n - seq_len - horizon + 1):
                seqs.append(Xd[i:i+seq_len])
                stats.append(Xs[i+seq_len-1])
                tgts.append(y[i+seq_len:i+seq_len+horizon])
        self.X_seq = torch.tensor(np.stack(seqs), dtype=torch.float32)
        self.X_stat = torch.tensor(np.stack(stats), dtype=torch.float32)
        self.y = torch.tensor(np.stack(tgts), dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_stat[idx], self.y[idx]

In [12]:
SEQ_LEN, HORIZON = 30, 30
BATCH_SIZE, LR, EPOCHS = 64, 1e-3, 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
# Unique store IDs
store_ids = df['zipcode'].unique()
np.random.shuffle(store_ids)

n = len(store_ids)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_ids = store_ids[:n_train]
val_ids   = store_ids[n_train:n_train + n_val]
test_ids  = store_ids[n_train + n_val:]

# Split the DataFrame
train_df = df[df.zipcode.isin(train_ids)]
val_df   = df[df.zipcode.isin(val_ids)]
test_df  = df[df.zipcode.isin(test_ids)]

print(f"Train stores: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

Train stores: 359 | Val: 77 | Test: 78


In [19]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(AdditiveAttention, self).__init__()
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)

    def forward(self, query, keys):
        q_proj = self.W_q(query).unsqueeze(1)
        k_proj = self.W_k(keys)
        scores = self.v(torch.tanh(q_proj + k_proj)).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)
        context = (weights.unsqueeze(-1) * keys).sum(dim=1)
        return context, weights

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dim):
        super(ScaledDotProductAttention, self).__init__()
        self.scale = dim ** 0.5

    def forward(self, query, key, value):
        scores = torch.bmm(query, key.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights, value)
        return context, weights

class HybridLSTMAttnWithStatic(nn.Module):
    def __init__(
        self,
        input_dim,
        static_dim,
        seq_len=30,
        lstm_units=128,
        dense_units=100,
        forecast_horizon=30,
        attention_type='multihead',
        num_heads=1,
        static_mlp_units=64
    ):
        super().__init__()
        self.seq_len = seq_len
        self.attn_type = attention_type

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=lstm_units, batch_first=True)

        if attention_type == 'multihead':
            self.attn = nn.MultiheadAttention(embed_dim=input_dim, num_heads=num_heads, batch_first=True)
            self.attn_out_dim = input_dim
        elif attention_type == 'additive':
            self.attn = AdditiveAttention(lstm_units)
            self.attn_out_dim = lstm_units
        elif attention_type == 'scaled_dot':
            self.attn = ScaledDotProductAttention(lstm_units)
            self.attn_out_dim = lstm_units
        else:
            raise ValueError(f"Unknown attn type {attention_type}")

        self.flat_in  = seq_len * input_dim
        self.flat_lstm= seq_len * lstm_units
        self.flat_attn= (attn_out_dim if attention_type=='additive' 
                         else seq_len * self.attn_out_dim)
        
        self.static_net = nn.Sequential(
            nn.Linear(static_dim, static_mlp_units),
            nn.ReLU(),
            nn.Linear(static_mlp_units, static_mlp_units),
            nn.ReLU()
        )
        self.flat_stat = static_mlp_units

        joint_dim = self.flat_in + self.flat_lstm + self.flat_attn + self.flat_stat
        self.fcdl = nn.Sequential(
            nn.Linear(joint_dim, dense_units),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(dense_units, forecast_horizon)

    def forward(self, x_seq, x_stat):
        lstm_out, (h_n, c_n) = self.lstm(x_seq)

        if self.attn_type == 'multihead':
            attn_out, _ = self.attn(x_seq, x_seq, x_seq)
            attn_flat = attn_out.reshape(x_seq.size(0), -1)
        elif self.attn_type == 'additive':
            query = h_n[-1]  # last layer’s hidden as query
            context, _ = self.attn(query, lstm_out)
            attn_flat = context
        else:
            context, _ = self.attn(lstm_out, lstm_out, lstm_out)
            attn_flat = context.reshape(x_seq.size(0), -1)

        x_flat    = x_seq.reshape(x_seq.size(0), -1)
        lstm_flat = lstm_out.reshape(x_seq.size(0), -1)

        stat_feat = self.static_net(x_stat)

        concat = torch.cat([x_flat, lstm_flat, attn_flat, stat_feat], dim=1)
        hidden = self.fcdl(concat)
        return self.output_layer(hidden)

In [20]:
model = HybridLSTMAttnWithStatic(len(DYNAMIC), len(STATIC_NUM), seq_len=SEQ_LEN).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

In [21]:
print(len(DYNAMIC))

90


In [22]:
from tqdm import tqdm

train_losses, val_losses = [], []

for ep in range(1, EPOCHS + 1):
    print(f"\n=== Epoch {ep}/{EPOCHS} ===")
    
    model.train()
    batch_losses = []
    train_bar = tqdm(train_loader, desc="Training", leave=False)
    for i, (xs, xb, yt) in enumerate(train_bar):
        xs, xb, yt = xs.to(DEVICE), xb.to(DEVICE), yt.to(DEVICE)
        opt.zero_grad()
        pred = model(xs, xb)
        loss = loss_fn(pred, yt)
        loss.backward()
        opt.step()
        
        batch_losses.append(loss.item())
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        if (i + 1) % 200 == 0:
            print(f"  [Batch {i+1}/{len(train_loader)}] train_loss={loss.item():.4f}")

    epoch_train_loss = np.mean(batch_losses)
    train_losses.append(epoch_train_loss)
    print(f" → Epoch {ep} Train MSE: {epoch_train_loss:.4f}")

    model.eval()
    val_batch_losses = []
    val_bar = tqdm(val_loader, desc="Validation", leave=False)
    with torch.no_grad():
        for i, (xs, xb, yt) in enumerate(val_bar):
            xs, xb, yt = xs.to(DEVICE), xb.to(DEVICE), yt.to(DEVICE)
            loss = loss_fn(model(xs, xb), yt)
            val_batch_losses.append(loss.item())
            val_bar.set_postfix(val_loss=f"{loss.item():.4f}")

    epoch_val_loss = np.mean(val_batch_losses)
    val_losses.append(epoch_val_loss)
    print(f" → Epoch {ep}   Val MSE: {epoch_val_loss:.4f}")

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xs, xb, yt in val_loader:
        p = model(xs.to(DEVICE), xb.to(DEVICE)).cpu().numpy()
        all_preds.append(p); all_true.append(yt.numpy())

all_preds = np.vstack(all_preds)
all_true = np.vstack(all_true)

mae  = mean_absolute_error(all_true, all_preds)
rmse = mean_squared_error(all_true, all_preds, squared=False)
print(f"\n*** Final Validation MAE: {mae:.4f}  |  RMSE: {rmse:.4f} ***")


=== Epoch 1/2 ===


Training:   7%|▋         | 203/2829 [00:08<01:45, 24.84it/s, loss=0.0156]

  [Batch 200/2829] train_loss=0.0141


Training:  14%|█▍        | 404/2829 [00:16<01:36, 25.19it/s, loss=0.0206]

  [Batch 400/2829] train_loss=0.0192


...

  [Batch 2600/2829] train_loss=0.0166


Training:  99%|█████████▉| 2804/2829 [02:15<00:01, 18.04it/s, loss=0.0193]

  [Batch 2800/2829] train_loss=0.0231


 → Epoch 2 Train MSE: 0.0198


 → Epoch 2   Val MSE: 0.0199

*** Final Validation MAE: 0.0819  |  RMSE: 0.1411 ***
